In [28]:
import sys

import pandas as pd
import nibabel as nib
import numpy as np
import scipy
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, losses
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

import os
import datetime

In [29]:
def generate_visit_pairs(path_to_csv):
    df = pd.read_csv(path_to_csv, delimiter=";")

    df['CDR'] = df['CDR'].astype(str).str.replace(',', '.')
    df['CDR'] = pd.to_numeric(df['CDR'], errors='coerce')

    # Ensure consistent sorting
    df = df.sort_values(by=['Subject ID', 'Visit'])

    pairs = []

    for id, group in df.groupby('Subject ID'):
        visits = group.sort_values(by='Visit').reset_index(drop=True)

        for i in range(len(visits) - 1):
            v0 = visits.iloc[i]
            v1 = visits.iloc[i + 1]

            cdr0 = v0['CDR']
            cdr1 = v1['CDR']

            # Create progression label
            # CDR rose
            if cdr1 > cdr0:
                label = 2
            # CDR stable
            elif cdr1 == cdr0:
                label = 1
            # CDR reverted
            elif cdr1 < cdr0:
                label = 0
            
            pairs.append((v0['MRI ID'], v1['MRI ID'], label))

    return pairs

def resize_and_load_img(hdr_path: str):
    img = nib.load(hdr_path)
    img_data = img.get_fdata()

    img_data = ((img_data - np.mean(img_data)) / np.std(img_data)).astype(np.float32)

    """
    plt.imshow(data[data.shape[0] // 2], cmap="gray")
    plt.show()
    """

    img_data = img_data[..., 0]

    zoom_factors = [t / c for t, c in zip((128, 128, 128, 1), img_data.shape)]
    volume = scipy.ndimage.zoom(img_data, zoom_factors, order=1)
    
    volume = volume[..., np.newaxis]
    
    return volume

pairs = generate_visit_pairs("./oasis_longitudinal_demographics.csv") # LaterDO: files from cmd
scans = []
labels = []

for scan1, scan2, label in pairs:

    img1 = resize_and_load_img("../Images/" + scan1 + "/RAW/mpr-1.nifti.hdr")
    img2 = resize_and_load_img("../Images/" + scan2 + "/RAW/mpr-1.nifti.hdr")
    
    pair = np.stack([img1, img2], axis=0)

    scans.append(pair)
    labels.append(label)

x = np.array(scans)
y = np.array(labels)

print("X's shape: " + str(x.shape))
print("y's shape: " + str(y.shape))

rand_indxs = list(range(x.shape[0]))
np.random.shuffle(rand_indxs)
x = x[rand_indxs]
y = y[rand_indxs]

val_offset = int(x.shape[0]*0.6)
test_offset = int(x.shape[0]*0.8)

x_train = x[:val_offset]
x_val = x[val_offset:test_offset]
x_test = x[test_offset:]

y_train = y[:val_offset]
y_val = y[val_offset:test_offset]
y_test = y[test_offset:]

X's shape: (223, 2, 128, 128, 128, 1)
y's shape: (223,)


In [34]:
log_dir = os.path.join("asd")
os.makedirs(log_dir, exist_ok=True)
tensorboard_callback = TensorBoard(log_dir=log_dir,
                                    histogram_freq=1,
                                    write_graph=True,
                                    write_images=True,
                                    update_freq='epoch')

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [35]:
def convnext_block(x, filters, kernel_size=3):
    # Lightweight ConvNeXt-style block for 3D
    shortcut = x
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)  # Depthwise
    x = layers.LayerNormalization()(x)
    x = layers.Conv3D(filters, 1, padding='same')(x)  # Pointwise

    shortcut = layers.Conv3D(filters=x.shape[-1], kernel_size=1, padding='same')(shortcut)
    x = layers.Add()([shortcut, x])
    return x

def feature_extractor(input_shape):
    inputs = Input(input_shape)
    x = layers.Conv3D(16, 3, padding='same', activation='gelu')(inputs)
    x = convnext_block(x, 16)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 32)
    x = layers.MaxPooling3D(2)(x)
    x = convnext_block(x, 64)
    x = layers.GlobalAveragePooling3D()(x)
    model = Model(inputs, x, name="feature_extractor")
    return model

def build_model():
    input_1 = Input((128, 128, 128, 1))
    input_2 = Input((128, 128, 128, 1))
    
    extractor = feature_extractor((128, 128, 128, 1))
    
    emb_1 = extractor(input_1)
    emb_2 = extractor(input_2)
    
    merged = layers.Concatenate()([emb_1, emb_2])
    x = layers.Dense(128, activation='gelu')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='gelu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(3, activation='softmax')(x)
    
    model = Model([input_1, input_2], output)
    return model


def data_generator(x_data, y_data, batch_size):
    dataset_size = len(x_data)
    indices = np.arange(dataset_size)

    while True:
        np.random.shuffle(indices)
        for start_idx in range(0, dataset_size, batch_size):
            end_idx = min(start_idx + batch_size, dataset_size)
            batch_indices = indices[start_idx:end_idx]
            
            batch_x1 = x_data[batch_indices, 0]
            batch_x2 = x_data[batch_indices, 1]
            batch_y = y_data[batch_indices]
            
            yield (batch_x1, batch_x2), batch_y

model = build_model()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

x1 = x[:, 0]
print("x1shape: " + str(x1.shape))
x2 = x[:, 1]
print("x2shape: " + str(x1.shape))

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_16      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 128, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_extractor   │ (None, 64)        │     85,360 │ input_layer_15[0… │
│ (Functional)        │                   │            │ input_layer_16[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 128)       │          0 │ feature_extracto… │
│ (Concatenate)       │                   │            │ feature_extracto… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 128)       │     16,512 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 128)       │          0 │ dense_15[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 64)        │      8,256 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 64)        │          0 │ dense_16[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 3)         │        195 │ dropout_11[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 110,323 (430.95 KB)

 Trainable params: 110,323 (430.95 KB)

 Non-trainable params: 0 (0.00 B)

x1shape: (223, 128, 128, 128, 1)
x2shape: (223, 128, 128, 128, 1)


In [39]:
print(os.getcwd())

model.fit(data_generator(x_train, y_train, batch_size=2),
    steps_per_epoch=len(x_train) // 2,
    validation_data=data_generator(x_val, y_val, batch_size=2),
    validation_steps=len(x_val) // 2, 
    epochs=50,
    callbacks=[early_stop, tensorboard_callback]
)

d:\Egyetem\Mester\KutMód\AD_class_resources\Code


FailedPreconditionError: {{function_node __wrapped__CreateSummaryFileWriter_device_/job:localhost/replica:0/task:0/device:CPU:0}} asd is not a directory [Op:CreateSummaryFileWriter] name: 